# 4 - Guide d'intégration des modèles de différents packages

## Importation des modules

In [1]:
# Importation des modules
# Modules de base
import numpy as np
import pandas as pd
import sys

# Ajout du chemin
sys.path.append('..')

# Importation des utilitaires sklearn
from sklearn.utils import _safe_indexing
from sklearn.utils.metaestimators import _safe_split
from sklearn.model_selection import cross_val_predict

# Importation des modèles
# Sklearn
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler
# XGBoost

# Sktime
from sktime.forecasting.arima import ARIMA
# Tslearn

# Darts


# Importation des pipelines
# Sklearn
from sklearn.pipeline import Pipeline

#  Eléments du package à intégrer
# Crossval
from tsforecast.crossvals import (
    PanelOutOfSampleSplit
)

## 1. Création de données synthétiques

In [2]:
# Fonction de génération des données de panel
def generate_panel_data(entities=['A', 'B', 'C'], start_date='2015-01-01', periods=120, 
                        freq='MS', heterogeneous_effects=True, common_trend=True, 
                        entity_specific_seasonality=True, cross_sectional_correlation=0.3,
                        missing_data_prob=0.0):
    """
    Generate synthetic panel data with various characteristics.
    
    Args:
        entities: List of entity identifiers
        start_date: Start date for the panel
        periods: Number of time periods per entity
        freq: Frequency of observations
        heterogeneous_effects: Whether entities have different baseline levels
        common_trend: Whether to include a common trend across entities
        entity_specific_seasonality: Whether seasonality patterns differ by entity
        cross_sectional_correlation: Correlation between entity shocks
        missing_data_prob: Probability of missing observations
    
    Returns:
        pd.DataFrame: Panel data with MultiIndex (entity, date)
    """
    # Création de l'index temporel
    dates = pd.date_range(start=start_date, periods=periods, freq=freq)
    
    # Création du MultiIndex (entity, date)
    index = pd.MultiIndex.from_product([entities, dates], names=['entity', 'date'])
    
    # Initialisation du DataFrame
    panel_data = pd.DataFrame(index=index)
    
    # Génération des effets fixes par entité (hétérogénéité)
    if heterogeneous_effects:
        entity_effects = {entity: np.random.normal(0, 2) for entity in entities}
    else:
        entity_effects = {entity: 0 for entity in entities}
    
    # Tendance commune
    if common_trend:
        common_trend_values = 0.02 * np.arange(periods)
    else:
        common_trend_values = np.zeros(periods)
    
    # Génération de chocs corrélés entre entités
    if cross_sectional_correlation > 0:
        # Chocs communs
        common_shocks = np.random.normal(0, 1, periods)
        # Chocs idiosyncratiques
        idiosyncratic_shocks = {
            entity: np.random.normal(0, 1, periods) 
            for entity in entities
        }
    
    # Construction des séries pour chaque entité
    values = []
    
    # Parcours des entités
    for entity in entities:
        # Effet fixe de l'entité
        entity_effect = entity_effects[entity]
        
        # Saisonnalité spécifique à l'entité
        if entity_specific_seasonality:
            # Période et amplitude différentes selon l'entité
            seasonal_period = 20 + hash(entity) % 40  # Entre 20 et 60
            seasonal_amplitude = 0.5 + (hash(entity) % 100) / 200  # Entre 0.5 et 1.0
        else:
            seasonal_period = 30
            seasonal_amplitude = 0.5
        
        seasonal_values = seasonal_amplitude * np.sin(2 * np.pi * np.arange(periods) / seasonal_period)
        
        # Processus autorégressif spécifique à l'entité
        ar_coef = 0.5 + (hash(entity) % 50) / 100  # Entre 0.5 et 1.0
        ar_process = np.zeros(periods)
        ar_process[0] = np.random.normal(0, 0.5)
        for t in range(1, periods):
            ar_process[t] = ar_coef * ar_process[t-1] + np.random.normal(0, 0.5)
        
        # Combinaison des composantes
        if cross_sectional_correlation > 0:
            # Chocs avec corrélation croisée
            correlated_shocks = (
                np.sqrt(cross_sectional_correlation) * common_shocks +
                np.sqrt(1 - cross_sectional_correlation) * idiosyncratic_shocks[entity]
            )
        else:
            correlated_shocks = np.random.normal(0, 1, periods)
        
        # Combinaison des valeurs
        entity_values = (
            entity_effect + 
            common_trend_values + 
            seasonal_values + 
            ar_process + 
            correlated_shocks
        )
        
        # Ajout de données manquantes
        if missing_data_prob > 0:
            missing_mask = np.random.random(periods) < missing_data_prob
            entity_values[missing_mask] = np.nan
        
        values.extend(entity_values)
    
    # Création du DataFrame final avec les valeurs
    panel_data['value'] = values
    
    # Ajout de variables explicatives
    panel_data['lag_value'] = panel_data.groupby('entity')['value'].shift(1)
    panel_data['trend'] = np.tile(np.arange(periods), len(entities))
    panel_data['month'] = panel_data.index.get_level_values('date').month
    
    return panel_data

In [3]:
# Génération de différents types de données de panel
print("📊 Génération de données de panel ...")

# Panel 1: Données équilibrées avec effets hétérogènes
entities_small = ['FR', 'DE', 'IT', 'ES']
df_panel = generate_panel_data(
    entities=entities_small,
    start_date='2015-01-01',
    periods=120,
    freq='MS',
    heterogeneous_effects=True,
    common_trend=True,
    entity_specific_seasonality=True,
    cross_sectional_correlation=0.4
)

# Suppression des Nan
df_panel.dropna(how='any', inplace=True)

print(f"✅ Génération de panels terminée:")
print(f"Caractéristiques des données générées : {df_panel.shape[0]} observations, {len(entities_small)} entités")

# Affichage des premières observations de chaque panel
print(f"\n📋 Aperçu des données:")
print(df_panel.head(10))

📊 Génération de données de panel ...
✅ Génération de panels terminée:
Caractéristiques des données générées : 476 observations, 4 entités

📋 Aperçu des données:
                      value  lag_value  trend  month
entity date                                         
FR     2015-02-01  2.588288   1.924363      1      2
       2015-03-01  2.847314   2.588288      2      3
       2015-04-01  3.701100   2.847314      3      4
       2015-05-01  3.569715   3.701100      4      5
       2015-06-01  4.812536   3.569715      5      6
       2015-07-01  4.768668   4.812536      6      7
       2015-08-01  3.600970   4.768668      7      8
       2015-09-01  4.699572   3.600970      8      9
       2015-10-01  7.601895   4.699572      9     10
       2015-11-01  5.826732   7.601895     10     11


## 2. Création d'un cadre de prévision à partir des éléments développpés dans `tsforecast`

In [4]:
# Séparation en X et y
y = df_panel['value'].copy()
X = df_panel.drop('value', axis=1)

# Initialisation de l'horizon de prédiction
horizon=2
# Initialisation du délai de publication
delays=1
# Application de l'horizon aux données afin d'aligner X et y à prévoir
# /!\ Créer une classe plus intelligente qui utilise la régularité de la série (sur données de panel et de séries temporelles) pour ajouter les dates manquantes aux extrémités de la période et ne pas perdre de données
X = X.shift(-horizon)

# Initialisation de la crossval pour l'ensemble des tests
cv = PanelOutOfSampleSplit(
    test_indices=['2024-01-01', '2024-02-01'], 
    test_size=1, 
    gap=horizon + delays
)
splits = list(cv.split(X, y))
# Extraction des indices d'entrainement et de test
train, test = splits[0]

# Séparation des données d'entrainement et de test
X_train, y_train = _safe_indexing(X, train), _safe_indexing(y, train)
X_test = _safe_indexing(X, test)

## 3. Intégration des modèles de différents packages dans un cadre unifié, compatible avec les éléments développés dans `tsforecast`

### 3.1. Intégration des modèles `sklearn`

L'intégration de l'ensemble des estimateurs de `sklearn` dans le workflow se fait nativement, les observations `X` et `y` étant déjà alignées. La syntaxe est ainsi celle de `sklearn` :
- `.fit(X,y)` pour l'entraînement ;
- `.predict(X)` pour la prédiction ;

L'ensemble des utilitaires comme `GridSearchCV`, `cross_val_score` etc ... peuvent être utilisés de la même manière en respectant la syntaxe originale du package.

In [5]:
# Importation du modèle
from sklearn.linear_model import LinearRegression

# Initialisation du modèle
estimator=LinearRegression()

# Entrainement du modèle
estimator.fit(X_train, y_train)
# Prédiction du modèle
y_pred_sklearn = estimator.predict(X_test)

y_pred_sklearn

array([ 4.45606452, -2.25431905,  4.80554148,  1.18904446])

L'intégration des `Pipeline` de `sklearn.pipeline`, permettant de combiner des transformations opérées sur les données et l'entraînement d'un estimateur en fin de processus, se fait de la même manière en respectant la syntaxe originale.

In [6]:
# Initialisation de la pipeline
estimator = Pipeline([
    ('StandardScalerTransformer', StandardScaler()),
    ('RidgeEstimator', LinearRegression())
])

# Entrainement du modèle
estimator.fit(X_train, y_train)
# Prédiction du modèle
y_pred_sklearn_pipeline = estimator.predict(X_test)

y_pred_sklearn_pipeline

array([ 4.45606452, -2.25431905,  4.80554148,  1.18904446])

### 3.2. Intégration des modèles `xgboost`

L'API de `xgboost` est entièrement compatible avec celle de `sklearn`, aussi ses estimateurs s'intègrent nativement dans un workflow similaire. La syntaxe est ainsi celle de `sklearn` :
- `.fit(X,y)` pour l'entraînement ;
- `.predict(X)` pour la prédiction ;

L'ensemble des utilitaires comme `GridSearchCV`, `cross_val_score` etc ... de `sklearn` peuvent être utilisés avec ces modèles sans modification de la syntaxe.

In [7]:
# Importation du modèle
from xgboost import XGBRegressor

# Initialisation du modèle
estimator=XGBRegressor()

# Entrainement du modèle
estimator.fit(X_train, y_train)
# Prédiction du modèle
y_pred_xgboost = estimator.predict(X_test)

y_pred_xgboost

array([ 4.596928 , -4.6577945,  6.1216044,  2.108076 ], dtype=float32)

 La combinaison de transformers avec les estimateurs de `xgboost` dans une `Pipeline` de `sklearn.pipeline` s'opère également sans difficulté.

In [8]:
# Importation du modèle
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

# Initialisation de la pipeline
estimator = Pipeline([
    ('StandardScalerTransformer', StandardScaler()),
    ('XGBoostEstimator', XGBRegressor())
])

# Entrainement du modèle
estimator.fit(X_train, y_train)
# Prédiction du modèle
y_pred_xgboost_pipeline = estimator.predict(X_test)

y_pred_xgboost_pipeline

array([ 4.596928 , -4.6577945,  6.1216044,  2.108076 ], dtype=float32)

### 3.3. Intégration des modèles `tslearn`

`tslearn` reprend également l'API de `sklearn`, aussi ses estimateurs s'intègrent nativement dans un workflow similaire. La syntaxe est ainsi celle de `sklearn` :
- `.fit(X,y)` pour l'entraînement ;
- `.predict(X)` pour la prédiction ;

L'ensemble des utilitaires comme `GridSearchCV`, `cross_val_score` etc ... de `sklearn` peuvent être utilisés avec ces modèles sans modification de la syntaxe.

In [9]:
# Importation du modèle
from tslearn.svm import TimeSeriesSVR

# Initialisation du modèle
estimator=TimeSeriesSVR()

# Entrainement du modèle
estimator.fit(X_train, y_train)
# Prédiction du modèle
y_pred_tslearn = estimator.predict(X_test)

y_pred_tslearn

/opt/python/lib/python3.13/site-packages/tslearn/bases/bases.py:16: UserWarning: h5py not installed, hdf5 features will not be supported.
Install h5py to use hdf5 features: http://docs.h5py.org/
  warn(h5py_msg)


array([ 3.20521737, -0.15878887,  3.37248109,  1.4555995 ])

La combinaison de ces modèles avec des transformers de `sklearn` dans une `Pipeline` de `sklearn.Pipeline` s'opère de manière transparente

In [10]:
# Importation du modèle
from sklearn.preprocessing import StandardScaler
from tslearn.svm import TimeSeriesSVR

# Initialisation de la pipeline
estimator = Pipeline([
    ('StandardScalerTransformer', StandardScaler()),
    ('SVREstimator', TimeSeriesSVR())
])

# Entrainement du modèle
estimator.fit(X_train, y_train)
# Prédiction du modèle
y_pred_tslearn_pipeline = estimator.predict(X_test)

y_pred_tslearn_pipeline

array([ 5.303361  , -1.40444062,  5.46081167,  1.73559139])

### 3.4. Intégration des modèles `sktime`

Les modèles issus de des modules  `sktime.regression`, `sktime.classification` suivent l'API de `sklearn` mais sont rarement pertinents car ils nécessitent le plus souvent la définition d'une `window` qui peut venir dégrader les performances du modèle. Cette intégration n'est pas prévue dans `tsforecast`. Il est souvent préférable d'utiliser l'équivalent dans `sklearn`.

Les transformers de `sktime.transformations` s'intègrent eux sans difficulté dans le workflow avec la syntaxe :
- `.fit(X,y)` pour l'entraînement ;
- `.transform(X)` pour la transform ;

L'ensemble des utilitaires comme `GridSearchCV`, `cross_val_score` etc ... de `sklearn` peuvent être utilisés avec ces transformers sans modification de la syntaxe.

Les forecasters ont une syntaxe qui diffère légèrement de celle des modèles du package `sklearn` en ce que les méthodes `fit` et `predict` ont également pour argument un horizon de prédiction. Si l'horizon de prédiction est spécifié lors de l'entrainement du modèle, il n'a pas besoin de l'être à nouveau pour la prévision. La syntaxe des `forecasters` est ainsi la suivante :
- `.fit(X,y, fh)` pour l'entraînement ;
- `.predict(X, fh)` pour la prédiction ;

#### 3.4.1 Utilisation des modèles de `sktime.forecasting`

L'horizon de prédiction étant déjà appliqué en faisant un `shift`/ `Lag` sur les données pour aligner les covariables `X` avec la valeur de `y` à prévoir, correspondant à un horizon de prédiction donné, il est inutile d'appliquer cette logique de décalage une nouvelle fois. Aussi, l'intégration dans la syntaxe `sklearn` des forecasters de sktime peut se faire en spécifiant `fh=0` comme argument de la méthode `fit` et en utilisant l'adapter `SktimeAdapter` qui permet de convertir les données `X` et `y` au format attendu par les forecasters.

Les utilitaires de `sklearn` peuvent s'intégrer dans cette syntaxe :
- `sklearn.pipeline.Pipeline` possède une méthode `fit(X, y=None, **params)` qui permet la spécification de `fh=0` qui est ensuite transmis comme argument de la méthode `fit` du forecaster. `sktime` a également implémenté sa version de cet utilitaire à travers `sktime.pipeline.Pipeline` qui possède la même signature qu'une pipeline `sklearn` et tolère estimateurs classiques et forecasters comme étapes ainsi que `sktime.forecasting.compose.ForecastingPipeline` qui possède la même syntaxe qu'forecaster classique pour ses méthodes `fit` et `predict`
- `sklearn.model_selection.GridSearchCV` possède également une méthode `fit(X, y=None, **params)` qui permet la spécification de `fh=0`. `sktime` a également implémenté sa version de cet utilitaire à travers `sktime.forecasting.model_selection.ForecastingGridSearchCV` qui possède la même signature qu'un forecaster classique pour ses méthodes `fit` et `predict`
- `sklearn.model_selection.cross_val_score` possède un argument `params` (`sklearn.model_selection.cross_val_predict`possède un argument `fit_params` qui poursuit le même objectif) qui permet de spécifier sous la forme d'un dictionnaire des paramètres de la méthode `fit` de l'estimateur. En l'occurrence `{'fh' : 0}` permet de spécifier le comportement attendu

In [11]:
# Importation du modèle
from sktime.forecasting.naive import NaiveForecaster
# Importation de l'adapter
from tsforecast.adapters import SktimeAdapter

# Initialisation du modèle
estimator=SktimeAdapter(forecaster=NaiveForecaster())

# Entraînement du modèle
estimator.fit(X_train, y_train)
# Prédiction du modèle
y_pred_forecaster_sktime = estimator.predict(X_test)

y_pred_forecaster_sktime

,,value
entity,date,
DE,2023-09-01,5.838341
ES,2023-09-01,-2.083041
FR,2023-09-01,6.060683
IT,2023-09-01,2.564350


In [12]:
# Importation de cross_val_score
from sklearn.model_selection import cross_val_score
# Importation du modèle
from sktime.forecasting.naive import NaiveForecaster

# Calcul du score
score_forecaster_sktime = cross_val_score(
    estimator=SktimeAdapter(forecaster=NaiveForecaster()),
    X=X,
    y=y,
    cv=cv,
    n_jobs=-1,
)

score_forecaster_sktime

array([0.83877861, 0.76230165])

`sklearn.pipeline.Pipeline` peut être utilisé avec des forecasters `sktime` en transmettant `fh=0` via les paramètres de fit.

In [13]:
# Importation de la pipeline
from sklearn.pipeline import Pipeline
# Importation du transformer
from sklearn.preprocessing import StandardScaler
# Importation du modèle
from sktime.forecasting.naive import NaiveForecaster
# Importation de l'adapter
from tsforecast.adapters import SktimeAdapter

# Construction de la pipeline avec preprocessing sklearn et forecaster sktime
pipeline_sklearn = Pipeline([
    ('scaler', StandardScaler()),  # Preprocessing sklearn
    ('forecaster', SktimeAdapter(forecaster=NaiveForecaster()))  # Forecaster sktime
])

# Entraînement de la pipeline avec fh=0
pipeline_sklearn.fit(X_train, y_train)

# Prédiction
y_pred_pipeline_sklearn_sktime = pipeline_sklearn.predict(X_test)

y_pred_pipeline_sklearn_sktime

,,value
entity,date,
DE,2023-09-01,5.838341
ES,2023-09-01,-2.083041
FR,2023-09-01,6.060683
IT,2023-09-01,2.564350


`GridSearchCV` de sklearn peut être utilisé pour optimiser les hyperparamètres d'un forecaster `sktime`.

In [14]:
# Importation de la gridsearch
from sklearn.model_selection import GridSearchCV
# Importation de la pipeline
from sklearn.pipeline import Pipeline
# Importation du transformer
from sklearn.preprocessing import StandardScaler
# Importation du modèle
from sktime.forecasting.naive import NaiveForecaster
# Importation de l'adapter
from tsforecast.adapters import SktimeAdapter

# Définition de la pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('forecaster', SktimeAdapter(forecaster=NaiveForecaster()))
])

# Définition de la grille de paramètres
param_grid = {
    'forecaster__forecaster__strategy': ["last", "mean", "drift"],
}

# Initialisation du GridSearchCV
grid_search_sklearn = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=cv,
    n_jobs=-1,
    scoring='neg_mean_squared_error'
)

# Entraînement avec fh=0
grid_search_sklearn.fit(X, y)

# Meilleurs paramètres et score
print(f"Meilleurs paramètres: {grid_search_sklearn.best_params_}")
print(f"Meilleur score: {grid_search_sklearn.best_score_}")

# Prédiction avec le meilleur modèle
y_pred_grid_sklearn = grid_search_sklearn.predict(X_test)

y_pred_grid_sklearn

Meilleurs paramètres: {'forecaster__forecaster__strategy': 'last'}
Meilleur score: -2.1737007395527304


,,value
entity,date,
FR,2024-12-01,5.281058
DE,2024-12-01,5.448135
IT,2024-12-01,1.228462
ES,2024-12-01,0.437017


#### 3.4.2. Utilisation des forecasters comme plateforme d'intégration des modèles de différents packages

`make_reduction` transforme n'importe quel régresseur (issu de `sklearn`, `xgboost` etc ...) en forecaster via des **stratégies de réduction** :
- `recursive`: Un modèle, réutilise prédictions pour multi-step (rapide, peut accumuler erreurs)
- `direct`: Un modèle par horizon (précis sur long terme, plus lent)
- `multioutput`: Un modèle prédit tous horizons simultanément

Le sliding window est automatique: `[y[t-window_length], ..., y[t-1]] → y[t]`. 

**Cependant `make_reduction` n'est pas compatible avec un horizon de prédiction nul (fh=0)**.

Il est cependant possible d'utiliser `YfromX` qui crée des forecasters où `X` prédit `y` directement **sans lags autorégressifs**.

**Différence clé:** `make_reduction` utilise des lags de `y` comme features. `YfromX` utilise `X` uniquement sans historique de `y`.

In [15]:
# Importation de XfromX
from sktime.forecasting.compose import YfromX
# Importation du modèle
from sklearn.ensemble import RandomForestRegressor
# Importation de l'adapter
from tsforecast.adapters import SktimeAdapter

# Création d'un forecaster qui utilise uniquement X (sans lags de y)
forecaster_rf_yfromx = SktimeAdapter(
    forecaster=YfromX(
        estimator=RandomForestRegressor(n_estimators=100, random_state=42)
    )
)

# Entraînement du modèle
# Ici, le modèle apprend directement la relation X -> y
forecaster_rf_yfromx.fit(X_train, y_train)

# Prédiction
y_pred_rf_yfromx = forecaster_rf_yfromx.predict(X_test)

y_pred_rf_yfromx

,,value
entity,date,
DE,2023-09-01,6.315649
ES,2023-09-01,-0.672590
FR,2023-09-01,5.686062
IT,2023-09-01,1.847762


### 3.5. Intégration des modèles `darts`

L'API de `darts` diffère de celle de `sklearn` en ce que les modèles manipulent des objets `TimeSeries` propres au package plutôt que des DataFrames pandas. Pour intégrer ces modèles dans un workflow unifié, le package `tsforecast` fournit un adapter `DartsAdapter` qui encapsule les modèles `darts` dans une interface compatible `sklearn`.

La syntaxe devient alors celle de `sklearn` :
- `.fit(X, y)` pour l'entraînement ;
- `.predict(X)` pour la prédiction ;
- `.score(X, y)` pour l'évaluation ;

L'adapter gère automatiquement la conversion entre pandas et TimeSeries de darts. L'ensemble des utilitaires `sklearn` comme `GridSearchCV`, `cross_val_score`, `Pipeline` etc. peuvent être utilisés de manière transparente.

In [16]:
# Importation de l'adapter
from tsforecast.adapters import DartsAdapter
# Importation d'un modèle darts
from darts.models import LinearRegressionModel

# Initialisation du modèle darts
darts_model = LinearRegressionModel(lags=5, lags_past_covariates=3)

# Encapsulation dans l'adapter
estimator = DartsAdapter(model=darts_model)

# Entrainement du modèle (syntaxe sklearn)
estimator.fit(X_train, y_train)

# Prédiction du modèle (syntaxe sklearn)
y_pred_darts = estimator.predict(X_test)

# Évaluation avec score R²
r2_score = estimator.score(X_test, _safe_indexing(y, test))
print(f"Score R²: {r2_score:.3f}")

y_pred_darts

Support for PyTorch based likelihood models not available. To enable them, install "darts[torch]" or "darts[all]" (with pip); or "u8darts-torch" or "u8darts-all" (with conda).
Support for Torch based models not available. To enable them, install "darts[torch]" or "darts[all]" (with pip); or "u8darts-torch" or "u8darts-all" (with conda).
The StatsForecast module could not be imported. To enable support for the AutoARIMA, AutoETS and Croston models, please consider installing it.
ValueError: The `past_covariates` are not long enough. Given horizon `n=0`, `min(lags_past_covariates)=-3`, `max(lags_past_covariates)=-1` and `output_chunk_length=1`, the `past_covariates` have to range from 2023-07-01 00:00:00 until 2023-09-01 00:00:00 (inclusive), but they only range from 2024-01-01 00:00:00 until 2024-01-01 00:00:00.


ValueError: The `past_covariates` are not long enough. Given horizon `n=0`, `min(lags_past_covariates)=-3`, `max(lags_past_covariates)=-1` and `output_chunk_length=1`, the `past_covariates` have to range from 2023-07-01 00:00:00 until 2023-09-01 00:00:00 (inclusive), but they only range from 2024-01-01 00:00:00 until 2024-01-01 00:00:00.

`GridSearchCV` de sklearn peut être utilisé pour optimiser les hyperparamètres des modèles darts encapsulés dans l'adapter.

In [ ]:
# Importation de GridSearchCV
from sklearn.model_selection import GridSearchCV
# Importation de l'adapter et du modèle
from tsforecast.adapters import DartsAdapter
from darts.models import LinearRegressionModel

# Définition de la pipeline avec modèle darts
pipeline_darts_grid = Pipeline([
    ('scaler', StandardScaler()),
    ('forecaster', DartsAdapter(model=LinearRegressionModel()))
])

# Définition de la grille de paramètres
# Note: Utiliser le préfixe 'forecaster__model__' pour accéder aux paramètres du modèle darts
param_grid = {
    'forecaster__model__lags': [3, 5, 7, 10],
    'forecaster__model__output_chunk_length': [1, 2, 3]
}

# Initialisation du GridSearchCV
grid_search_darts = GridSearchCV(
    estimator=pipeline_darts_grid,
    param_grid=param_grid,
    cv=cv,
    n_jobs=-1,
    scoring='neg_mean_squared_error'
)

# Entraînement
grid_search_darts.fit(X, y)

# Meilleurs paramètres et score
print(f"Meilleurs paramètres: {grid_search_darts.best_params_}")
print(f"Meilleur score: {grid_search_darts.best_score_:.4f}")

# Prédiction avec le meilleur modèle
y_pred_grid_darts = grid_search_darts.predict(X_test)

y_pred_grid_darts

L'adapter `DartsAdapter` s'intègre naturellement dans les `Pipeline` de `sklearn.pipeline`, permettant de combiner des transformations sklearn avec des modèles darts.

In [ ]:
# Importation de la pipeline
from sklearn.pipeline import Pipeline
# Importation du transformer
from sklearn.preprocessing import StandardScaler
# Importation de l'adapter et du modèle darts
from tsforecast.adapters import DartsAdapter
from darts.models import ExponentialSmoothing

# Construction de la pipeline avec preprocessing sklearn et modèle darts
pipeline_darts = Pipeline([
    ('scaler', StandardScaler()),  # Preprocessing sklearn
    ('forecaster', DartsAdapter(model=ExponentialSmoothing()))  # Modèle darts encapsulé
])

# Entraînement de la pipeline
pipeline_darts.fit(X_train, y_train)

# Prédiction
y_pred_pipeline_darts = pipeline_darts.predict(X_test)

y_pred_pipeline_darts

### 3.6. Intégration des modèles de `hierarchicalforecast`

Le package `hierarchicalforecast` se spécialise dans la réconciliation de prévisions hiérarchiques (géographiques, temporelles, etc.). Son API nécessite des formats de données spécifiques et des méthodes particulières. 

Pour intégrer ces fonctionnalités dans un workflow unifié, `tsforecast` fournit l'adapter `HierarchicalForecastAdapter` qui encapsule les méthodes de réconciliation dans une interface compatible `sklearn`.

La syntaxe devient alors celle de `sklearn` :
- `.fit(X, y)` pour construire la hiérarchie à partir des données historiques ;
- `.predict(X)` pour réconcilier des prévisions de base ;
- `.score(X, y)` pour évaluer la qualité de la réconciliation ;

L'adapter gère automatiquement :
- La conversion entre DataFrames pandas avec MultiIndex et le format hierarchicalforecast
- L'agrégation des séries selon la hiérarchie spécifiée
- La construction de la matrice de sommation (S matrix)
- L'application des méthodes de réconciliation

**Note:** Les exemples ci-dessous utilisent des hiérarchies simples pour illustration. Pour des hiérarchies cross-sectionnelles réelles (ex: Pays > Région > Ville), il faudrait des données avec un MultiIndex approprié.

In [ ]:
# Importation de l'adapter
from tsforecast.adapters import HierarchicalForecastAdapter
# Importation des méthodes de réconciliation
from hierarchicalforecast.methods import BottomUp, MinTrace

# Pour cet exemple, nous utilisons une hiérarchie temporelle simple
# qui agrège les données à différents niveaux temporels
spec_temporal = {
    'quarterly': 3,  # Agrégation sur 3 mois
    'monthly': 1,    # Niveau de base (mensuel)
}

# Initialisation de l'adapter avec méthodes de réconciliation
adapter_hierarchical = HierarchicalForecastAdapter(
    reconcilers=[BottomUp(), MinTrace(method='ols')],
    spec=spec_temporal,
    aggregation_type='local'
)

# Entraînement sur données historiques (construction de la hiérarchie)
adapter_hierarchical.fit(None, y_train)

# Obtention d'informations sur la hiérarchie construite
hierarchy_info = adapter_hierarchical.get_hierarchy_info()
print(f"Nombre total de séries: {hierarchy_info['n_series']}")
print(f"Séries au niveau de base: {hierarchy_info['n_bottom']}")
print(f"Niveaux de hiérarchie: {hierarchy_info['levels']}")

# Pour la réconciliation, on aurait besoin de prévisions de base
# Ici on utilise les valeurs de test comme "prévisions" pour démonstration
y_hat_base = _safe_indexing(y, test)

# Réconciliation des prévisions
y_reconciled = adapter_hierarchical.predict(y_hat_base)

print(f"\nFormes des données:")
print(f"Prévisions de base: {y_hat_base.shape}")
print(f"Prévisions réconciliées: {y_reconciled.shape}")

y_reconciled.head()

L'adapter peut être utilisé avec `cross_val_score` pour évaluer la performance de différentes méthodes de réconciliation.

In [ ]:
# Importation de cross_val_score
from sklearn.model_selection import cross_val_score
# Importation de l'adapter et méthodes
from tsforecast.adapters import HierarchicalForecastAdapter
from hierarchicalforecast.methods import BottomUp

# Définition de l'adapter
adapter_hierarchical_cv = HierarchicalForecastAdapter(
    reconcilers=[BottomUp()],
    spec={'quarterly': 3, 'monthly': 1}
)

# Évaluation avec cross-validation
# Note: Ceci évalue la capacité de la réconciliation à améliorer les prévisions
scores_hierarchical = cross_val_score(
    estimator=adapter_hierarchical_cv,
    X=y,  # Les "prévisions de base" sont les vraies valeurs pour cet exemple
    y=y,  # Les valeurs cibles
    cv=cv,
    scoring='r2'
)

print(f"Scores R² par fold: {scores_hierarchical}")
print(f"Score R² moyen: {scores_hierarchical.mean():.4f} (+/- {scores_hierarchical.std():.4f})")

scores_hierarchical

### 3.7. Intégration des modèles de `opera`

Le package `opera` se spécialise dans l'agrégation en ligne d'experts (online expert aggregation), permettant de combiner dynamiquement les prévisions de plusieurs modèles. Son API spécifique nécessite des formats particuliers.

Pour intégrer ces fonctionnalités dans un workflow unifié, `tsforecast` fournit l'adapter `OperaAdapter` qui encapsule les algorithmes d'agrégation dans une interface compatible `sklearn`.

La syntaxe devient alors celle de `sklearn` :
- `.fit(X, y)` pour entraîner le système d'agrégation sur les prédictions des experts ;
- `.predict(X)` pour combiner les prédictions des experts ;
- `.partial_fit(X, y)` pour mise à jour incrémentale (online learning) ;
- `.score(X, y)` pour évaluer la qualité de l'ensemble ;

L'adapter gère automatiquement :
- La conversion des formats de données
- L'initialisation des coefficients d'experts
- La mise à jour des poids selon les performances passées
- Le calcul des prédictions d'ensemble

**Cas d'usage typique:** Vous avez entraîné plusieurs modèles différents (ex: Ridge, Lasso, RandomForest) et vous voulez les combiner de manière optimale en fonction de leurs performances.

In [ ]:
# Importation de l'adapter
from tsforecast.adapters import OperaAdapter
# Importation de modèles sklearn pour créer des "experts"
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor

# Étape 1: Entraîner plusieurs modèles "experts"
print("Entraînement des modèles experts...")
expert_ridge = Ridge(alpha=1.0).fit(X_train, y_train)
expert_lasso = Lasso(alpha=1.0).fit(X_train, y_train)
expert_rf = RandomForestRegressor(n_estimators=50, random_state=42).fit(X_train, y_train)

# Étape 2: Obtenir leurs prédictions (qui serviront d'input à opera)
expert_predictions_train = np.column_stack([
    expert_ridge.predict(X_train),
    expert_lasso.predict(X_train),
    expert_rf.predict(X_train)
])

expert_predictions_test = np.column_stack([
    expert_ridge.predict(X_test),
    expert_lasso.predict(X_test),
    expert_rf.predict(X_test)
])

print(f"Prédictions des experts (train): {expert_predictions_train.shape}")
print(f"Prédictions des experts (test): {expert_predictions_test.shape}")

# Étape 3: Entraîner l'adapter Opera pour combiner les experts
estimator_opera = OperaAdapter(
    model="BOA",  # Bernstein Online Aggregation
    loss_type="mse"
)

# Entraînement sur les prédictions des experts
estimator_opera.fit(expert_predictions_train, y_train.values)

# Étape 4: Combiner les prédictions des experts sur le test
y_pred_opera = estimator_opera.predict(expert_predictions_test)

# Évaluation
r2_opera = estimator_opera.score(expert_predictions_test, _safe_indexing(y, test).values)
print(f"\nScore R² de l'ensemble: {r2_opera:.4f}")

# Comparaison avec les experts individuels
from sklearn.metrics import r2_score
print(f"Score R² Ridge: {r2_score(_safe_indexing(y, test), expert_ridge.predict(X_test)):.4f}")
print(f"Score R² Lasso: {r2_score(_safe_indexing(y, test), expert_lasso.predict(X_test)):.4f}")
print(f"Score R² RandomForest: {r2_score(_safe_indexing(y, test), expert_rf.predict(X_test)):.4f}")

y_pred_opera

`GridSearchCV` peut être utilisé pour optimiser le choix de l'algorithme d'agrégation et du type de perte.

In [ ]:
# Importation de GridSearchCV
from sklearn.model_selection import GridSearchCV
from tsforecast.adapters import OperaAdapter

# Préparation des prédictions des experts sur l'ensemble complet
expert_predictions_full = np.column_stack([
    expert_ridge.predict(X),
    expert_lasso.predict(X),
    expert_rf.predict(X)
])

# Définition de la grille de paramètres
param_grid = {
    'model': ['BOA', 'EWA', 'MLpol'],  # Différents algorithmes d'agrégation
    'loss_type': ['mse', 'mae'],  # Différentes fonctions de perte
}

# Initialisation du GridSearchCV
grid_search_opera = GridSearchCV(
    estimator=OperaAdapter(),
    param_grid=param_grid,
    cv=cv,
    n_jobs=-1,
    scoring='neg_mean_squared_error'
)

# Entraînement
print("Recherche des meilleurs hyperparamètres...")
grid_search_opera.fit(expert_predictions_full, y.values)

# Meilleurs paramètres et score
print(f"\nMeilleurs paramètres: {grid_search_opera.best_params_}")
print(f"Meilleur score: {grid_search_opera.best_score_:.4f}")

# Prédiction avec le meilleur modèle
y_pred_grid_opera = grid_search_opera.predict(expert_predictions_test)

# Comparaison des performances
r2_grid = r2_score(_safe_indexing(y, test), y_pred_grid_opera)
print(f"Score R² (meilleur modèle): {r2_grid:.4f}")

y_pred_grid_opera

Une particularité d'`OperaAdapter` est le support de l'apprentissage incrémental via `partial_fit`, permettant de mettre à jour les poids des experts au fur et à mesure que de nouvelles données arrivent (online learning).

In [ ]:
# Simulation d'un scénario d'apprentissage incrémental
# Divisons les données d'entraînement en batches

# Initialisation de l'adapter
estimator_opera_online = OperaAdapter(model="EWA", loss_type="mse")  # Exponentially Weighted Average

# Premier batch: entraînement initial
batch_size = len(X_train) // 3
expert_preds_batch1 = expert_predictions_train[:batch_size]
y_batch1 = y_train.values[:batch_size]

print(f"Entraînement initial sur {batch_size} observations...")
estimator_opera_online.fit(expert_preds_batch1, y_batch1)

# Batches suivants: mise à jour incrémentale
for i in range(1, 3):
    start_idx = i * batch_size
    end_idx = (i + 1) * batch_size if i < 2 else len(X_train)
    
    expert_preds_batch = expert_predictions_train[start_idx:end_idx]
    y_batch = y_train.values[start_idx:end_idx]
    
    print(f"Mise à jour incrémentale {i}: {len(y_batch)} observations...")
    estimator_opera_online.partial_fit(expert_preds_batch, y_batch)

# Prédiction finale
y_pred_opera_online = estimator_opera_online.predict(expert_predictions_test)

# Évaluation
r2_online = estimator_opera_online.score(expert_predictions_test, _safe_indexing(y, test).values)
print(f"\nScore R² (apprentissage incrémental): {r2_online:.4f}")

y_pred_opera_online